## Análise de clientes

- Realize a limpeza das categorias de produtos para evitar duplicidade por erro de grafia.

In [1]:
import pandas as pd
import duckdb

# Vendas
df_vendas = pd.read_csv('../datasets/vendas_2023_2024.csv')

# Produtos com limpeza de categoria
df_produtos = pd.read_csv('../datasets/produtos_raw.csv')

def normalizar_categoria(valor):
    v = valor.replace(' ', '').lower()
    v = v.replace('ô', 'o').replace('ã', 'a').replace('ç', 'c')
    if 'eletro' in v:
        return 'eletronicos'
    elif 'prop' in v:
        return 'propulsao'
    elif 'ancor' in v or 'encor' in v:
        return 'ancoragem'
    return 'desconhecido'

df_produtos['category'] = df_produtos['actual_category'].apply(normalizar_categoria)
df_produtos = df_produtos.rename(columns={'code': 'id_product'})[['id_product', 'category']]

In [2]:
query = """
-- Ticket médio e diversidade por cliente
WITH base AS (
    SELECT
        v.id_client,
        v.id,
        v.total,
        v.qtd,
        p.category
    FROM df_vendas v
    LEFT JOIN df_produtos p ON p.id_product = v.id_product
),

metricas AS (
    SELECT
        id_client,
        SUM(total)                          AS faturamento_total,
        COUNT(id)                           AS frequencia,
        ROUND(SUM(total) / COUNT(id), 2)    AS ticket_medio,
        COUNT(DISTINCT category)            AS diversidade_categorias
    FROM base
    GROUP BY id_client
),

-- Top 10 clientes fiéis (diversidade >= 3, maior ticket médio, desempate por id_client)
top10 AS (
    SELECT *
    FROM metricas
    WHERE diversidade_categorias >= 3
    ORDER BY ticket_medio DESC, id_client ASC
    LIMIT 10
),

-- Categoria mais vendida em qtd para esses 10 clientes
categoria_top AS (
    SELECT
        b.category,
        SUM(b.qtd) AS total_itens
    FROM base b
    INNER JOIN top10 t ON t.id_client = b.id_client
    GROUP BY b.category
    ORDER BY total_itens DESC
    LIMIT 1
)

-- Resultado final: top 10 + categoria mais vendida
SELECT
    t.*,
    c.category  AS categoria_mais_vendida,
    c.total_itens
FROM top10 t
CROSS JOIN categoria_top c
ORDER BY t.ticket_medio DESC, t.id_client ASC
"""

df_resultado = duckdb.query(query).df()
df_resultado

,id_client,faturamento_total,frequencia,ticket_medio,diversidade_categorias,categoria_mais_vendida,total_itens
0,47,67668333.05,202,334991.75,4,propulsao,6429.0
1,42,74021826.35,232,319059.60,4,propulsao,6429.0
2,46,63296508.15,203,311805.46,4,propulsao,6429.0
3,9,72697344.85,234,310672.41,4,propulsao,6429.0
4,36,69349933.20,230,301521.45,4,propulsao,6429.0
5,2,67717055.25,225,300964.69,4,propulsao,6429.0
6,22,59663694.40,200,298318.47,4,propulsao,6429.0
7,28,65619098.85,220,298268.63,4,propulsao,6429.0
8,26,66653100.20,224,297558.48,4,propulsao,6429.0
9,5,61071262.70,210,290815.54,4,propulsao,6429.0


In [3]:
query_validacao = """
WITH base AS (
    SELECT v.id_client, v.qtd, p.category
    FROM df_vendas v
    LEFT JOIN df_produtos p ON p.id_product = v.id_product
),
top10 AS (
    SELECT id_client
    FROM (
        SELECT
            id_client,
            SUM(total) / COUNT(id)       AS ticket_medio,
            COUNT(DISTINCT category)     AS diversidade
        FROM base b2
        LEFT JOIN df_vendas v2 ON v2.id_client = b2.id_client
        GROUP BY id_client
    )
    WHERE diversidade >= 3
    ORDER BY ticket_medio DESC, id_client ASC
    LIMIT 10
)
SELECT
    b.category,
    SUM(b.qtd) AS total_itens
FROM base b
INNER JOIN top10 t ON t.id_client = b.id_client
GROUP BY b.category
ORDER BY total_itens DESC
"""

# Forma mais simples e direta de validar
top10_ids = duckdb.query("""
    SELECT id_client
    FROM (
        SELECT
            v.id_client,
            SUM(v.total) / COUNT(v.id)          AS ticket_medio,
            COUNT(DISTINCT p.category)           AS diversidade
        FROM df_vendas v
        LEFT JOIN df_produtos p ON p.id_product = v.id_product
        GROUP BY v.id_client
    )
    WHERE diversidade >= 3
    ORDER BY ticket_medio DESC, id_client ASC
    LIMIT 10
""").df()['id_client'].tolist()

df_top10_vendas = df_vendas[df_vendas['id_client'].isin(top10_ids)].merge(df_produtos, on='id_product')

categoria_mais_vendida = (
    df_top10_vendas.groupby('category')['qtd']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={'qtd': 'total_itens'})
)

print("Categoria mais vendida para os Top 10 clientes:")
categoria_mais_vendida

Categoria mais vendida para os Top 10 clientes:


,category,total_itens
0,propulsao,6429
1,ancoragem,6003
2,eletronicos,4809
3,desconhecido,567



**METODOLOGIA — Questão 5**

1. LIMPEZA DAS CATEGORIAS
   - Removidos espaços internos e convertido para minúsculo
   - Acentos normalizados (ô→o, ã→a, ç→c) para evitar falsos negativos
   - Mapeamento por raiz da palavra: 'eletro' → eletronicos,
     'prop' → propulsao, 'ancor'/'encor' → ancoragem
   - Cobre todos os 39 valores únicos encontrados no dataset

2. FILTRO DE DIVERSIDADE MÍNIMA
   - COUNT(DISTINCT category) por cliente após o join com produtos
   - Apenas clientes com diversidade >= 3 entram no ranking
   - Aplicado antes do ORDER BY e LIMIT 10

3. CONTAGEM DE ITENS RESTRITA AO TOP 10
   - Os 10 ids selecionados são isolados em uma CTE (top10)
   - O join de base com top10 garante que apenas transações
     desses clientes entrem no SUM(qtd) da categoria
   - Nenhuma venda de fora do grupo contamina o resultado